# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EimanZahra1472/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Ranking / Scoring.** The question this lane answers is "which pages should an editor review first?"  that's a "which ones first" question, which the framing skill maps directly to ranking/scoring, with priority score as the target and precision@K as the metric. There's a classification model underneath (predicting decline probability), but the deliverable that actually gets used is the ordered list, not a raw yes/no.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**The proxy** I'm using is is_declining_label = (trend_direction == "down"), matching the starter pipeline. Honest flag, per the framing skill's rule: this is a defined label, not a purely observed one  trend_direction is a bucket computed by thresholding trend_pct, so in a strict sense the model would partly be learning "what crosses this threshold" rather than a genuinely independent future outcome. It's not as bad as feeding trend_pct in directly as a feature (that's outright leakage, which I already saw firsthand in Week 2), but it's still a rule-derived proxy, not a future-window observed decline.

For this framing assignment I'm using it as-is because it's what the starter data supports without extra window-building. But the honest next step  and what I'd move toward for the capstone  is a genuinely observed label: prior 90 days of features → decline measured over the next 30 days, so the label is a real future outcome rather than a bucket computed from the current window.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Precision@50, with Precision@20 as a secondary check. I can compute this today on the baseline: the starter pipeline's hand-rule scores Precision@50 = 0.240. That's my "good" bar to beat  a learned ranking needs to clearly outperform 0.240 at the top of the list to be worth using over the fixed rule. I'll validate with client-holdout (not random row) splits, since pages from the same client could let a model just memorize client-specific patterns rather than learn transferable signal.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [2]:
!git clone https://github.com/EimanZahra1472/flyrank-ml-internship-starter.git
%cd flyrank-ml-internship-starter

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 132, done.
remote: Counting objects: 100% (132/132), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 132 (delta 42), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (132/132), 1.87 MiB | 13.13 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/content/flyrank-ml-internship-starter


In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]

print("Unit of analysis: one row = one content page")
print(f"Shape: {df.shape}")
df[["content_id", "client_id", "impressions_90d", "days_since_last_update",
    "avg_position", "ctr", "content_age_days", "trend_direction"]].head(10)


Unit of analysis: one row = one content page
Shape: (30000, 44)


,content_id,client_id,impressions_90d,days_since_last_update,avg_position,ctr,content_age_days,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,20,10.6,0.76,187,down
1,content_a1fb4e703a9e,client_4e07408562,15320,25,20.3,0.05,445,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,20,36.5,0.09,141,down
3,content_331d6c4de07b,client_19581e27de,11751,22,6.2,0.49,463,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,14,44.0,0.13,263,down
5,content_d4084a4bc775,client_f369cb89fc,3970,20,8.5,0.03,147,down
6,content_9a34b442b552,client_8722616204,20,20,7.0,0.00,90,down
7,content_a63219c6e95a,client_19581e27de,1724,22,21.2,0.06,445,stable
8,content_5e6c160719bc,client_6208ef0f77,32574,20,46.0,0.09,90,down
9,content_c27558df2b0c,client_19581e27de,1240,104,4.9,0.16,257,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

For a content reviewer with limited weekly capacity, deciding which pages to review first for refresh, we will build a ranked priority queue from the starter content dataset, scoring is_declining_label (a rule-derived proxy for now, future-window decline eventually), measured by Precision@50. A wrong call costs either wasted reviewer time on a healthy page or a missed decline that compounds unnoticed. A plain rule isn't enough because the signal is spread across multiple interacting features, not one threshold  proven by the 0.240 → 0.740 precision gap between the baseline and the model. We will claim only observed, directional, decision-support results.

In [4]:
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(df["is_declining_label"].value_counts())
print(f"\nDeclining rate: {df['is_declining_label'].mean():.3f}")
df[["content_id", "trend_direction", "is_declining_label"]].head(10)

is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Declining rate: 0.542


,content_id,trend_direction,is_declining_label
0,content_304f48230142,down,1
1,content_a1fb4e703a9e,down,1
2,content_9aa793d4d895,down,1
3,content_331d6c4de07b,stable,0
4,content_d99b7a2d90ca,down,1
5,content_d4084a4bc775,down,1
6,content_9a34b442b552,down,1
7,content_a63219c6e95a,stable,0
8,content_5e6c160719bc,down,1
9,content_c27558df2b0c,down,1


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.